In [65]:
import json
import math
import os
import re
from typing import Dict, Tuple, Optional
from datetime import datetime
import copy
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import numpy as np


def lambda_handler(event, context):
    ## Get lambda inputs
    effective_date = datetime.strptime(event.get("effective_date"), "%Y-%m-%d").date()
    calculation_step = event.get("calculation_step", {})

    # ------------------------------------------------------------
    # STEP 1 : INPUT DATA  (inchangé)
    # ------------------------------------------------------------
    if calculation_step == "input_data":
        claims_file = "aggregated_data_example.xlsx"
        inflation_file = "med_inflation.xlsx"
        claims_df = pd.read_excel(claims_file)
        inflation_df = pd.read_excel(inflation_file)

        claims_df['claims_date '] = pd.to_datetime(claims_df['claims_date '], dayfirst=True, errors='coerce')
        claims_df['claims_year'] = claims_df['claims_date '].dt.year

        inflation_df = inflation_df.rename(columns={inflation_df.columns[0]: 'block_name'})
        inflation_long = inflation_df.melt(id_vars='block_name', var_name='Year', value_name='InflationRate')
        inflation_long['Year'] = inflation_long['Year'].astype(int)
        inflation_long['InflationRate'] = (
            inflation_long['InflationRate']
            .astype(str)
            .str.replace('%', '', regex=False)
            .str.replace(',', '.', regex=False)
            .astype(float) / 100
        )

        merged = pd.merge(
            claims_df,
            inflation_long,
            how='left',
            left_on=['block_name', 'claims_year'],
            right_on=['block_name', 'Year']
        )

        merged['adjusted_amount'] = merged['aggregated_amount_paid'] * (1 + merged['InflationRate'])
        final_df = merged[['block_name', 'claims_year', 'adjusted_amount']].copy()

        result_dict = {}
        for _, row in final_df.iterrows():
            block = row['block_name']
            year = int(row['claims_year']) if not pd.isna(row['claims_year']) else None
            adjusted = None if pd.isna(row['adjusted_amount']) else float(row['adjusted_amount'])
            entry = {"claims_year": year, "adjusted_amount": adjusted}
            result_dict.setdefault(block, []).append(entry)

        return result_dict

    # ------------------------------------------------------------
    # STEP 2 : CLAIMS PRICE PROJECTION (avec pondération)
    # ------------------------------------------------------------
    if calculation_step == "claims_price_projection":
        data = event.get("input_step_2", {})
        result = {}

        for block, values in data.items():
            df = pd.DataFrame(values)

            # Nettoyage
            df = df.dropna(subset=["adjusted_amount", "exposure"])
            if "weight" not in df.columns:
                df["weight"] = 1  # poids par défaut s’il manque

            # Calcul du coût par tête
            df["adjusted_amount_per_head"] = df["adjusted_amount"] / df["exposure"]
            df = df.sort_values("claims_year")

            # === Régression linéaire pondérée ===
            X = df["claims_year"].values.reshape(-1, 1)
            y = df["adjusted_amount_per_head"].values
            w = df["weight"].values

            model = LinearRegression()
            model.fit(X, y, sample_weight=w)

            y_pred = model.predict(X)

            # R² pondéré : calculé manuellement
            y_mean_w = np.average(y, weights=w)
            ss_tot = np.sum(w * (y - y_mean_w) ** 2)
            ss_res = np.sum(w * (y - y_pred) ** 2)
            r2 = 1 - ss_res / ss_tot if ss_tot != 0 else 0.0

            # Projection linéaire sur l’année suivante
            next_year = df["claims_year"].max() + 1
            projected_linear = float(model.predict(np.array([[next_year]]))[0])

            # === Moyenne pondérée ===
            projected_mean = float(np.average(df["adjusted_amount_per_head"], weights=w))

            result[block] = {
                "years": df["claims_year"].tolist(),
                "adjusted_amount_per_head": df["adjusted_amount_per_head"].round(2).tolist(),
                "weights": df["weight"].round(2).tolist(),
                "regression": {
                    "intercept": float(model.intercept_),
                    "slope": float(model.coef_[0]),
                    "r2_weighted": round(r2, 3),
                },
                "projection": {
                    "next_year": int(next_year),
                    "projection_linear_weighted": projected_linear,
                    "projection_mean_weighted": projected_mean,
                },
            }

        return result


### Call lambda step 1 : Input data

In [66]:
event = {"effective_date" : "2026-01-01", "calculation_step" : "input_data" }
result_step_1 = lambda_handler ( event, {})

### Call lambda step 2 : Calculation

In [67]:
input_step_2 =  {
    'Inpatient': [
        {'claims_year': 2025, 'adjusted_amount': 142332.75466, 'exposure': 1000, 'included': True, 'weight': 0.3333333333},
        {'claims_year': 2024, 'adjusted_amount': 232237.85799999998, 'exposure': 1000, 'included': True, 'weight': 0.3333333333},
        {'claims_year': 2023, 'adjusted_amount': 309762.7462, 'exposure': 1000, 'included': True, 'weight': 0.3333333333}
    ],
    'Outpatient': [
        {'claims_year': 2025, 'adjusted_amount': 517634.45391, 'exposure': 1000, 'included': True, 'weight': 0.3333333333},
        {'claims_year': 2024, 'adjusted_amount': 500593.13289999997, 'exposure': 1000, 'included': True, 'weight': 0.3333333333},
        {'claims_year': 2023, 'adjusted_amount': 450314.99999999994, 'exposure': 1000, 'included': True, 'weight': 0.3333333333}
    ],
    'Maternity': [
        {'claims_year': 2025, 'adjusted_amount': 7383.0996000000005, 'exposure': 1000, 'included': True, 'weight': 0.3333333333},
        {'claims_year': 2024, 'adjusted_amount': 127850.11959999999, 'exposure': 1000, 'included': True, 'weight': 0.3333333333},
        {'claims_year': 2023, 'adjusted_amount': 346781.9444, 'exposure': 1000, 'included': True, 'weight': 0.3333333333}
    ],
    'Dental': [
        {'claims_year': 2025, 'adjusted_amount': 35773.2312, 'exposure': 1000, 'included': True, 'weight': 0.3333333333},
        {'claims_year': 2024, 'adjusted_amount': 30069.018, 'exposure': 1000, 'included': True, 'weight': 0.3333333333},
        {'claims_year': 2023, 'adjusted_amount': 23016.1, 'exposure': 1000, 'included': True, 'weight': 0.3333333333}
    ]
}



In [68]:
event = {"effective_date" : "2026-01-01", "calculation_step" : "claims_price_projection" , "input_step_2" : input_step_2 }
result_step_2 = lambda_handler ( event, {})

In [69]:
result_step_2["Maternity"]

{'years': [2023, 2024, 2025],
 'adjusted_amount_per_head': [346.78, 127.85, 7.38],
 'weights': [0.33, 0.33, 0.33],
 'regression': {'intercept': 343632.3026587998,
  'slope': -169.69942239999992,
  'r2_weighted': 0.973},
 'projection': {'next_year': 2026,
  'projection_linear_weighted': -178.72712360002333,
  'projection_mean_weighted': 160.67172119999998}}

In [56]:
169667.26255809993 + 2026*-83.71499576999996

60.68112808000296

In [ ]:
result_step_2